In [3]:
"""
Generic MySQL Agent with phidata (SELECT + DML/DDL w/ safety) — with VISIBLE step-by-step prints
+ SQL Query Quality Scoring Tool (rubric-based)

Ported to: Ollama (gemma3:1b)

Deps:
  phidata
  SQLAlchemy>=2.0
  PyMySQL>=1.1.0
  python-dotenv

Env:
  (see .env snippet in chat)
"""

from dotenv import load_dotenv
load_dotenv()

import os
import re
from typing import Optional, List, Any

# ---- phidata ---------------------------------------------------------------
from phi.agent import Agent
from phi.tools import Toolkit
from phi.model.message import Message
from phi.model.ollama import Ollama  # <-- CHANGED (was OpenAIChat)

# ---- SQLAlchemy ------------------------------------------------------------
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.engine import Engine


# ====================== Helpers ============================================

_SELECT_START   = re.compile(r"(?is)^\s*select\b")
_WRITE_START    = re.compile(r"(?is)^\s*(insert|update|delete|merge|replace)\b")
_DDL_START      = re.compile(r"(?is)^\s*(create|alter|drop|truncate|rename)\b")
_LIMIT_PRESENT  = re.compile(r"(?is)\blimit\s+\d+(\s*,\s*\d+)?\b")

def _response_text(resp) -> str:
    """
    Convert model response to plain string safely across model backends.
    """
    if resp is None:
        return ""
    if isinstance(resp, str):
        return resp

    content = getattr(resp, "content", None)
    if isinstance(content, str):
        return content

    msg = getattr(resp, "message", None)
    if msg is not None:
        msg_content = getattr(msg, "content", None)
        if isinstance(msg_content, str):
            return msg_content

    return str(resp)


def _mysql_uri_from_env() -> str:
    host = os.getenv("MYSQL_HOST", "localhost")
    port = int(os.getenv("MYSQL_PORT", "3306"))
    user = os.getenv("MYSQL_USER")
    pwd  = os.getenv("MYSQL_PASSWORD")
    db   = os.getenv("MYSQL_DB")
    if not all([user, pwd, db]):
        raise RuntimeError("Set MYSQL_USER, MYSQL_PASSWORD, and MYSQL_DB in env")
    return f"mysql+pymysql://{user}:{pwd}@{host}:{port}/{db}"

def _get_engine() -> Engine:
    return create_engine(_mysql_uri_from_env(), pool_pre_ping=True)

def _render_schema(engine: Engine, tables: Optional[List[str]] = None) -> str:
    ins = inspect(engine)
    all_tables = sorted(ins.get_table_names())
    target_tables = all_tables if not tables else [t for t in tables if t in all_tables]
    if not target_tables:
        return "No tables found."

    lines: List[str] = []
    for t in target_tables:
        lines.append(f"# {t}")
        for col in ins.get_columns(t):
            colname = col.get("name")
            coltype = str(col.get("type"))
            nullable = col.get("nullable")
            default = col.get("default")
            lines.append(f"- {colname}: {coltype}, NULL={nullable}, DEFAULT={default}")
    head = "\n".join([f"- {t}" for t in all_tables[:50]])
    lines.append("\n# tables snapshot (truncated):\n" + head)
    return "\n".join(lines)

def _classify_sql(sql: str) -> str:
    s = (sql or "").strip()
    if not s:
        return "empty"
    if ";" in s:
        return "multi"
    if _SELECT_START.match(s):
        return "select"
    if _WRITE_START.match(s):
        return "write"
    if _DDL_START.match(s):
        return "ddl"
    return "other"

def _maybe_inject_limit(sql: str, row_cap: int) -> str:
    if _SELECT_START.match(sql) and not _LIMIT_PRESENT.search(sql):
        return f"{sql} LIMIT {row_cap}"
    return sql


def _build_ollama_model() -> Ollama:
    """
    Centralized Ollama model config.
    """
    model_id = os.getenv("OLLAMA_MODEL", "qwen3:1.7b")
    host = os.getenv("OLLAMA_HOST")  # optional
    timeout_raw = os.getenv("OLLAMA_TIMEOUT")
    timeout = int(timeout_raw) if timeout_raw else None

    # You can pass Ollama options here if you want:
    # options = {"temperature": 0.2, "num_ctx": 4096}
    options = None

    kwargs: dict[str, Any] = {"id": model_id}
    if host:
        kwargs["host"] = host
    if timeout is not None:
        kwargs["timeout"] = timeout
    if options is not None:
        kwargs["options"] = options

    return Ollama(**kwargs)


# ====================== Custom Toolkit (with explicit prints) ===============

class MySQLTools(Toolkit):
    """
    Tools:
      - mysql_schema(table_list: str="") -> str
      - mysql_run_sql(sql: str) -> str
      - mysql_nl2sql(question: str) -> str
      - mysql_answer(question: str) -> str
      - mysql_review_sql(sql: str) -> str
      - mysql_nl2sql_review(question: str) -> str
    """
    def __init__(self, model: Optional[Any] = None, row_cap: Optional[int] = None):
        super().__init__(name="mysql_tools")
        self.engine = _get_engine()
        self.model = model or _build_ollama_model()   # <-- CHANGED
        self.row_cap = row_cap or int(os.getenv("MYSQL_MAX_ROWS", "200"))
        self.allow_write = os.getenv("ALLOW_WRITE", "false").lower() == "true"
        self.run_explain = os.getenv("MYSQL_RUN_EXPLAIN", "false").lower() == "true"

        self.register(self.mysql_schema)
        self.register(self.mysql_run_sql)
        self.register(self.mysql_nl2sql)
        self.register(self.mysql_answer)
        self.register(self.mysql_review_sql)
        self.register(self.mysql_nl2sql_review)

    def mysql_schema(self, table_list: str = "") -> str:
        print("[tool] mysql_schema called", flush=True)
        tables = [t.strip() for t in table_list.split(",") if t.strip()] if table_list else None
        try:
            return _render_schema(self.engine, tables)
        except Exception as e:
            return f"Schema error: {e}"

    def mysql_run_sql(self, sql: str) -> str:
        print("[tool] mysql_run_sql called", flush=True)
        if not sql or not sql.strip():
            return "Please provide a SQL statement."
        sql = re.sub(r";.*$", "", sql, flags=re.S).strip()  # single statement

        kind = _classify_sql(sql)

        if kind in ("write", "ddl") and not self.allow_write:
            return ("Write/DDL blocked by safety policy. "
                    "Set ALLOW_WRITE=true to enable. The statement was classified as "
                    f"'{kind}'.\nSQL (blocked): {sql}")

        if kind == "select":
            sql = _maybe_inject_limit(sql, self.row_cap)

        print(f"[exec] Kind={kind} | SQL={sql}", flush=True)
        try:
            with self.engine.begin() as conn:
                result = conn.execute(text(sql))
                if kind == "select":
                    rows = result.fetchmany(self.row_cap + 1)
                    headers = list(result.keys())
                    truncated = len(rows) > self.row_cap
                    rows = rows[:self.row_cap]
                    out = []
                    header_line = " | ".join(map(str, headers))
                    out.append(header_line)
                    out.append("-" * max(3, len(header_line)))
                    for r in rows:
                        out.append(" | ".join("" if v is None else str(v) for v in r))
                    if truncated:
                        out.append(f"...(truncated at {self.row_cap} rows)")
                    return "\n".join(out)
                else:
                    try:
                        rowcount = result.rowcount
                    except Exception:
                        rowcount = None
                    return f"OK ({kind}). Rows affected: {rowcount}."
        except Exception as e:
            return f"Execution error ({kind}): {e}"

    def mysql_nl2sql(self, question: str) -> str:
        print("[tool] mysql_nl2sql called", flush=True)
        if not question or not question.strip():
            return "Please provide a question."

        try:
            schema_text = _render_schema(self.engine, None)
        except Exception as e:
            return f"Failed to load schema: {e}"

        system = (
            "You are a senior data engineer. Produce ONE valid MySQL statement that "
            "answers the user's request using only existing tables/columns from the schema. "
            "Be concise and correct. No comments/prose. No trailing ';'."
        )
        user = f"Database schema:\n{schema_text}\n\nRequest:\n{question}\n\nMySQL:"

        try:
            resp = self.model.response(messages=[
                Message(role="system", content=system),
                Message(role="user", content=user),
            ])
            sql = _response_text(resp).strip()
        except Exception as e:
            return f"NL2SQL error: {e}"

        sql = re.sub(r";.*$", "", sql, flags=re.S).strip()
        print(f"[nl2sql] SQL generated: {sql}", flush=True)
        if not sql:
            return "Failed to generate SQL. Try rephrasing."
        return sql

    def mysql_answer(self, question: str) -> str:
        print("[tool] mysql_answer called", flush=True)
        sql = self.mysql_nl2sql(question)
        if not sql or "error" in sql.lower() or "failed" in sql.lower():
            return sql
        return self.mysql_run_sql(sql)

    def mysql_review_sql(self, sql: str) -> str:
        print("[tool] mysql_review_sql called", flush=True)

        if not sql or not sql.strip():
            return "Please provide a SQL query to review."

        raw = sql.strip()

        if ";" in raw:
            first = re.split(r";", raw, maxsplit=1)[0].strip()
            multi_note = (
                "WARNING: Multi-statement SQL detected. Review performed ONLY on the first "
                "statement; multi-statement SQL is high-risk in production.\n"
            )
            sql_to_review = first
        else:
            multi_note = ""
            sql_to_review = raw

        kind = _classify_sql(sql_to_review)

        explain_text = ""
        if self.run_explain and kind == "select":
            try:
                with self.engine.connect() as conn:
                    exp = conn.execute(text(f"EXPLAIN FORMAT=JSON {sql_to_review}"))
                    row = exp.fetchone()
                    if row:
                        explain_text = str(row[0])
            except Exception as e:
                explain_text = f"EXPLAIN failed: {e}"

        system_prompt = (
            "You are a senior database performance engineer with expertise in SQL "
            "optimization and production systems.\n\n"
            "Review the SQL query below and evaluate its overall quality, safety, and performance.\n\n"
            "Scoring Scale: 0 = Extremely dangerous / guaranteed performance issues\n"
            "50 = Acceptable but significant optimization needed\n"
            "80 = Good production-ready query with minor improvements\n"
            "100 = Excellent, highly optimized, scalable, safe query\n\n"
            "Evaluate Across These Categories (Score Each 0–20):\n\n"
            "1) Index Utilization\n"
            "2) Performance Efficiency\n"
            "3) Scalability\n"
            "4) Concurrency & Locking Safety\n"
            "5) Production Safety\n\n"
            "Assume: Engine: InnoDB; Large production dataset (millions+ rows); High concurrency.\n\n"
            "Output Format:\n"
            "- Category scores (each /20)\n"
            "- Final total score (/100)\n"
            "- Top 3 weaknesses\n"
            "- Top 3 strengths\n"
            "- Most impactful improvement recommendation\n\n"
            "Do not include any extra sections."
        )

        user_prompt = (
            f"{multi_note}"
            f"SQL kind detected: {kind}\n\n"
            f"SQL:\n{sql_to_review}\n\n"
        )
        if explain_text:
            user_prompt += f"EXPLAIN FORMAT=JSON output:\n{explain_text}\n\n"

        try:
            resp = self.model.response(messages=[
                Message(role="system", content=system_prompt),
                Message(role="user", content=user_prompt),
            ])
            review = _response_text(resp).strip()
        except Exception as e:
            return f"SQL review error: {e}"

        return (multi_note + review) if (multi_note and review) else review

    def mysql_nl2sql_review(self, question: str) -> str:
        print("[tool] mysql_nl2sql_review called", flush=True)
        sql = self.mysql_nl2sql(question)
        if not sql or "error" in sql.lower() or "failed" in sql.lower():
            return sql
        review = self.mysql_review_sql(sql)
        return f"Generated SQL:\n{sql}\n\n---\n\n{review}"


# ====================== Agent builder =======================================

def build_agent(model_name: Optional[str] = None) -> Agent:
    """
    Same policy, but model is Ollama(id="gemma3:1b") by default.
    """
    # If caller passes model_name, treat it as an Ollama model id.
    model_id = model_name or os.getenv("OLLAMA_MODEL", "qwen3:1.7b")
    host = os.getenv("OLLAMA_HOST")

    model_kwargs: dict[str, Any] = {"id": model_id}
    if host:
        model_kwargs["host"] = host

    model = Ollama(**model_kwargs)

    tools = [MySQLTools(model=model)]

    instructions = (
        "You are a MySQL operations and analytics assistant.\n"
        "- If the user provides explicit SQL and asks to review/score/quality-check it, call `mysql_review_sql`.\n"
        "- If the user provides explicit SQL and asks to run it, call `mysql_run_sql`.\n"
        "- If the user asks in natural language and wants SQL + review (score), call `mysql_nl2sql_review`.\n"
        "- If the user asks in natural language and wants results, call `mysql_answer`.\n"
        "- If they ask about tables/columns, call `mysql_schema`.\n"
        "- Keep answers concise. Always return the DB result or a clear error.\n"
        "- Assume single-statement execution. Do not fabricate results."
    )

    return Agent(
        model=model,
        tools=tools,
        show_tool_calls=True,
        markdown=True,
        instructions=instructions,
    )


# ====================== Trace printer =======================================

def pretty_print_run(run_obj) -> None:
    print("\n=== Agent Run Trace ===", flush=True)
    msgs = getattr(run_obj, "messages", None)
    if not msgs:
        print(getattr(run_obj, "content", ""), flush=True)
        print("=== End Trace ===\n", flush=True)
        return
    for m in msgs:
        role = getattr(m, "role", "") or getattr(m, "type", "")
        content = getattr(m, "content", "")
        name = getattr(m, "name", "")
        if role == "tool":
            print(f"[tool:{name}] {content}", flush=True)
        else:
            label = role or "assistant"
            print(f"[{label}] {content}", flush=True)
    print("=== End Trace ===\n", flush=True)


# ====================== Minimal demo ========================================

if __name__ == "__main__":
    import time

    print("[demo] Generic MySQL Agent starting…", flush=True)
    try:
        engine = _get_engine()
        with engine.connect() as c:
            c.execute(text("SELECT 1"))
        print("[demo] DB connection OK.", flush=True)
    except Exception as e:
        print(f"[demo] DB connection failed: {e}", flush=True)
        raise SystemExit(1)

    agent = build_agent()
    tools: MySQLTools = agent.tools[0]  # type: ignore

    start = time.time()

    #user_input = "provide me description & counts of all columns of the table datasets and its significance"
    #user_input="provide me  distinct metric and its value each experiment"
    user_input="This is a mlflow database, there are experiments stored. provide me the distinct tags from the table tags for experiment id 1"
    print(f"[demo] Prompt: {user_input}\n", flush=True)

    print("[demo] Deterministic path: NL -> SQL -> EXECUTE", flush=True)
    sql_generated = tools.mysql_nl2sql(user_input)
    print(f"[demo] Generated SQL:\n{sql_generated}\n", flush=True)

    print("[demo] Deterministic path: SQL -> REVIEW (NO execution)", flush=True)
    review = tools.mysql_review_sql(sql_generated)
    print(f"[demo] Review:\n{review}\n", flush=True)

    print(f"\n[demo] Done in {time.time()-start:.2f}s", flush=True)


[demo] Generic MySQL Agent starting…
[demo] DB connection OK.
[demo] Prompt: This is a mlflow database, there are experiments stored. provide me the distinct tags from the table tags for experiment id 1

[demo] Deterministic path: NL -> SQL -> EXECUTE
[tool] mysql_nl2sql called
[nl2sql] SQL generated: ```sql
SELECT DISTINCT `tag` FROM registered_tags WHERE `experiment_id` = 1
```
[demo] Generated SQL:
```sql
SELECT DISTINCT `tag` FROM registered_tags WHERE `experiment_id` = 1
```

[demo] Deterministic path: SQL -> REVIEW (NO execution)
[tool] mysql_review_sql called
[demo] Review:
- Category scores: 10 / 20
- Final total score: 10 / 100
- Top 3 weaknesses:
  - Lack of Indexing: The query performs a full table scan, which is inefficient for a large dataset.
  - Missing Index:  There's no index on the `experiment_id` column, limiting the database's ability to efficiently filter the data.
- Top 3 strengths:
  - Simple and straightforward query.
- Most impactful improvement recommendation: